In [ ]:
!pip install albumentations

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from glob import glob
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, LearningRateScheduler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
from datetime import datetime
import time
import albumentations as A

# ===============================
# 1. CONFIGURATION
# ===============================
IMG_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 100
INITIAL_LEARNING_RATE = 1e-4
VALIDATION_SPLIT = 0.2
WEIGHT_DECAY = 1e-5

# CRITICAL FIX: Keep mixed precision for memory, but fix NaN in loss
USE_MIXED_PRECISION = True

USE_TTA = True
TTA_AUGMENTATIONS = 8

# Dataset paths
TRAIN_IMG_PATH = '/kaggle/input/warm-up-program-ai-vietnam-skin-segmentation/Train/Train/Image'
TRAIN_MASK_PATH = '/kaggle/input/warm-up-program-ai-vietnam-skin-segmentation/Train/Train/Mask'
TEST_IMG_PATH = '/kaggle/input/warm-up-program-ai-vietnam-skin-segmentation/Test/Test/Image'

# ===============================
# 2. ENHANCED DATA AUGMENTATION
# ===============================
def get_training_augmentation():
    """Enhanced augmentation pipeline using Albumentations"""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.5),
        A.OneOf([
            A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
            A.GridDistortion(p=0.5),
            A.OpticalDistortion(distort_limit=1, shift_limit=0.5, p=0.5),
        ], p=0.3),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
            A.GaussianBlur(blur_limit=(3, 7), p=0.5),
            A.MedianBlur(blur_limit=5, p=0.5),
        ], p=0.3),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.RandomGamma(gamma_limit=(80, 120), p=0.5),
            A.CLAHE(clip_limit=4.0, p=0.5),
        ], p=0.5),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.3),
    ])

def load_image(path, target_size=(IMG_SIZE, IMG_SIZE)):
    """Load and preprocess image with histogram equalization"""
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Failed to load image: {path}")
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size)
    
    # Apply CLAHE for better contrast
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    
    img = img.astype(np.float32) / 255.0
    return img

def load_mask(path, target_size=(IMG_SIZE, IMG_SIZE)):
    """Load and preprocess binary mask"""
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Failed to load mask: {path}")
    
    mask = cv2.resize(mask, target_size)
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)
    return mask

def load_data(img_dir, mask_dir):
    """Load all images and masks"""
    if not os.path.exists(img_dir):
        raise FileNotFoundError(f"Image directory not found: {img_dir}")
    if not os.path.exists(mask_dir):
        raise FileNotFoundError(f"Mask directory not found: {mask_dir}")
    
    img_paths = sorted(glob(os.path.join(img_dir, '*')))
    mask_paths = sorted(glob(os.path.join(mask_dir, '*')))
    
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    img_paths = [p for p in img_paths if os.path.splitext(p)[1].lower() in valid_extensions]
    mask_paths = [p for p in mask_paths if os.path.splitext(p)[1].lower() in valid_extensions]
    
    print(f"Found {len(img_paths)} images and {len(mask_paths)} masks")
    
    images, masks = [], []
    start_time = time.time()
    
    for idx, (img_path, mask_path) in enumerate(zip(img_paths, mask_paths)):
        if (idx + 1) % max(1, len(img_paths) // 10) == 0:
            progress = (idx + 1) / len(img_paths) * 100
            print(f"Progress: {progress:.0f}% ({idx+1}/{len(img_paths)})")
        
        images.append(load_image(img_path))
        masks.append(load_mask(mask_path))
    
    print(f"Data loading complete in {time.time() - start_time:.2f}s")
    return np.array(images), np.array(masks)

# ===============================
# 3. ATTENTION UNET ARCHITECTURE
# ===============================
def attention_gate(x, g, inter_channels):
    """Attention gate for focusing on relevant features"""
    theta_x = layers.Conv2D(inter_channels, 1, padding='same')(x)
    phi_g = layers.Conv2D(inter_channels, 1, padding='same')(g)
    
    concat = layers.Add()([theta_x, phi_g])
    act = layers.Activation('relu')(concat)
    psi = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(act)
    
    return layers.Multiply()([x, psi])

def conv_block(inputs, num_filters, dropout_rate=0.3, use_batchnorm=True):
    """Convolutional block with dropout and batch normalization"""
    x = layers.Conv2D(num_filters, 3, padding='same', 
                      kernel_regularizer=l2(WEIGHT_DECAY))(inputs)
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    
    x = layers.Conv2D(num_filters, 3, padding='same',
                      kernel_regularizer=l2(WEIGHT_DECAY))(x)
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    return x

def encoder_block(inputs, num_filters, dropout_rate=0.3):
    """Encoder block"""
    x = conv_block(inputs, num_filters, dropout_rate)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, num_filters, dropout_rate=0.3, use_attention=True):
    """Decoder block with attention gates"""
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(inputs)
    
    if use_attention:
        skip_features = attention_gate(skip_features, x, num_filters // 2)
    
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, num_filters, dropout_rate)
    return x

def attention_unet_model(input_size=(IMG_SIZE, IMG_SIZE, 3)):
    """Build Attention UNet model"""
    inputs = layers.Input(input_size)
    
    # Encoder
    s1, p1 = encoder_block(inputs, 64, dropout_rate=0.1)
    s2, p2 = encoder_block(p1, 128, dropout_rate=0.2)
    s3, p3 = encoder_block(p2, 256, dropout_rate=0.3)
    s4, p4 = encoder_block(p3, 512, dropout_rate=0.4)
    
    # Bottleneck
    b1 = conv_block(p4, 1024, dropout_rate=0.5)
    
    # Decoder with attention
    d1 = decoder_block(b1, s4, 512, dropout_rate=0.4, use_attention=True)
    d2 = decoder_block(d1, s3, 256, dropout_rate=0.3, use_attention=True)
    d3 = decoder_block(d2, s2, 128, dropout_rate=0.2, use_attention=True)
    d4 = decoder_block(d3, s1, 64, dropout_rate=0.1, use_attention=True)
    
    # Output
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(d4)
    
    model = models.Model(inputs, outputs, name='Attention_UNet')
    return model

# ===============================
# 4. IMPROVED LOSS FUNCTIONS (FIXED FOR NUMERICAL STABILITY)
# ===============================
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice coefficient - FIXED: Cast to float32 to prevent NaN"""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice = (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
    return dice

def weighted_dice_loss(y_true, y_pred, foreground_weight=2.0, smooth=1e-6):
    """Weighted Dice loss - prioritizes lesion pixels"""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    
    # Weight foreground (lesions) more than background
    weights = tf.where(tf.equal(y_true_f, 1.0), foreground_weight, 1.0)
    
    intersection = tf.reduce_sum(weights * y_true_f * y_pred_f)
    denominator = tf.reduce_sum(weights * y_true_f) + tf.reduce_sum(weights * y_pred_f)
    
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return 1.0 - dice

def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-6):
    """Focal Tversky loss - handles class imbalance better"""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    
    true_pos = tf.reduce_sum(y_true_f * y_pred_f)
    false_neg = tf.reduce_sum(y_true_f * (1.0 - y_pred_f))
    false_pos = tf.reduce_sum((1.0 - y_true_f) * y_pred_f)
    
    tversky = (true_pos + smooth) / (true_pos + alpha * false_neg + beta * false_pos + smooth)
    focal_tversky = tf.pow((1.0 - tversky), gamma)
    
    return focal_tversky

def combined_loss(y_true, y_pred):
    """Combined loss: Weighted Dice + Focal Tversky"""
    dice_loss = weighted_dice_loss(y_true, y_pred, foreground_weight=2.0)
    tversky_loss = focal_tversky_loss(y_true, y_pred)
    return 0.5 * dice_loss + 0.5 * tversky_loss

def iou_metric(y_true, y_pred, smooth=1e-6):
    """IoU metric - FIXED: Cast to float32"""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou

# ===============================
# 5. LEARNING RATE SCHEDULER
# ===============================
def cosine_decay_with_warmup(epoch, total_epochs=EPOCHS, warmup_epochs=5):
    """Cosine decay with warmup"""
    if epoch < warmup_epochs:
        return INITIAL_LEARNING_RATE * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return INITIAL_LEARNING_RATE * 0.5 * (1 + np.cos(np.pi * progress))

# ===============================
# 6. DATA GENERATOR WITH AUGMENTATION
# ===============================
class DataGenerator(keras.utils.Sequence):
    """Custom data generator with albumentations"""
    def __init__(self, images, masks, batch_size, augmentation=None, shuffle=True):
        self.images = images
        self.masks = masks
        self.batch_size = batch_size
        self.augmentation = augmentation
        self.shuffle = shuffle
        self.indices = np.arange(len(self.images))
        self.on_epoch_end()
    
    def __len__(self):
        return len(self.images) // self.batch_size
    
    def __getitem__(self, index):
        indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_images = []
        batch_masks = []
        
        for i in indices:
            image = self.images[i]
            mask = self.masks[i]
            
            if self.augmentation:
                augmented = self.augmentation(image=image, mask=mask.squeeze())
                image = augmented['image']
                mask = augmented['mask']
                mask = np.expand_dims(mask, axis=-1)
            
            batch_images.append(image)
            batch_masks.append(mask)
        
        return np.array(batch_images), np.array(batch_masks)
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# ===============================
# 7. POST-PROCESSING
# ===============================
def post_process_mask(mask, min_size=150, kernel_size=5):
    """Enhanced post-processing"""
    # Convert to uint8
    mask = (mask * 255).astype(np.uint8)
    
    # Apply morphological closing to fill small holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # Apply median filter
    mask = cv2.medianBlur(mask, 5)
    
    # Remove small components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] < min_size:
            mask[labels == i] = 0
    
    # Threshold back to binary
    mask = (mask > 127).astype(np.uint8)
    return mask

# ===============================
# 8. TEST-TIME AUGMENTATION
# ===============================
def predict_with_tta(model, image, n_augmentations=TTA_AUGMENTATIONS):
    """TTA with multiple augmentations"""
    predictions = []
    
    # Original
    pred = model.predict(np.expand_dims(image, axis=0), verbose=0)[0, :, :, 0]
    predictions.append(pred)
    
    # Horizontal flip
    img_flipped_h = np.fliplr(image)
    pred = model.predict(np.expand_dims(img_flipped_h, axis=0), verbose=0)[0, :, :, 0]
    predictions.append(np.fliplr(pred))
    
    # Vertical flip
    img_flipped_v = np.flipud(image)
    pred = model.predict(np.expand_dims(img_flipped_v, axis=0), verbose=0)[0, :, :, 0]
    predictions.append(np.flipud(pred))
    
    # Both flips
    img_flipped_hv = np.flipud(np.fliplr(image))
    pred = model.predict(np.expand_dims(img_flipped_hv, axis=0), verbose=0)[0, :, :, 0]
    predictions.append(np.fliplr(np.flipud(pred)))
    
    # Rotations
    for k in range(1, 4):
        img_rotated = np.rot90(image, k)
        pred = model.predict(np.expand_dims(img_rotated, axis=0), verbose=0)[0, :, :, 0]
        predictions.append(np.rot90(pred, -k))
    
    # Transpose
    img_transposed = np.transpose(image, (1, 0, 2))
    pred = model.predict(np.expand_dims(img_transposed, axis=0), verbose=0)[0, :, :, 0]
    predictions.append(np.transpose(pred, (1, 0)))
    
    return np.mean(predictions, axis=0)

# ===============================
# 9. RLE ENCODING
# ===============================
def mask2rle(mask):
    """Convert binary mask to run-length encoding"""
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

# ===============================
# 10. MAIN TRAINING FUNCTION
# ===============================
def main_training():
    """Main training pipeline"""
    print("\n" + "=" * 70)
    print("IMPROVED SKIN LESION SEGMENTATION - TRAINING PIPELINE")
    print("=" * 70)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nConfiguration:")
    print(f"  Image Size: {IMG_SIZE}x{IMG_SIZE}")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Initial LR: {INITIAL_LEARNING_RATE}")
    print(f"  Mixed Precision: {USE_MIXED_PRECISION} (with float32 loss - prevents NaN)")
    print(f"  TTA: {USE_TTA}")
    
    # Enable mixed precision
    if USE_MIXED_PRECISION:
        from tensorflow.keras import mixed_precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print("  Mixed precision policy set: float16 compute, float32 loss")
    
    # Load data
    print("\n" + "=" * 70)
    print("LOADING DATA")
    print("=" * 70)
    X_train_full, y_train_full = load_data(TRAIN_IMG_PATH, TRAIN_MASK_PATH)
    print(f"Loaded {len(X_train_full)} training samples")
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=VALIDATION_SPLIT,
        random_state=42
    )
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    
    # Create data generators
    train_gen = DataGenerator(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        augmentation=get_training_augmentation(),
        shuffle=True
    )
    
    # Build model
    print("\n" + "=" * 70)
    print("BUILDING ATTENTION UNET MODEL")
    print("=" * 70)
    model = attention_unet_model(input_size=(IMG_SIZE, IMG_SIZE, 3))
    print(f"Model built - Total parameters: {model.count_params():,}")
    
    # Compile model
    print("\n" + "=" * 70)
    print("COMPILING MODEL")
    print("=" * 70)
    model.compile(
        optimizer=Adam(learning_rate=INITIAL_LEARNING_RATE),
        loss=combined_loss,
        metrics=[dice_coefficient, iou_metric]
    )
    print("Model compiled with:")
    print("  Loss: Combined (Weighted Dice + Focal Tversky)")
    print("  Optimizer: Adam with cosine decay + warmup")
    
    # Setup callbacks
    callbacks = [
        ModelCheckpoint(
            'best_attention_unet.h5',
            monitor='val_dice_coefficient',
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        LearningRateScheduler(cosine_decay_with_warmup, verbose=1),
        EarlyStopping(
            monitor='val_dice_coefficient',
            patience=20,
            mode='max',
            restore_best_weights=True,
            verbose=1
        )
    ]
    
    # Train model
    print("\n" + "=" * 70)
    print("TRAINING MODEL")
    print("=" * 70)
    
    history = model.fit(
        train_gen,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n" + "=" * 70)
    print("TRAINING COMPLETED!")
    print("=" * 70)
    print(f"Best Val Dice: {max(history.history['val_dice_coefficient']):.6f}")
    print(f"Best Val IoU: {max(history.history['val_iou_metric']):.6f}")
    
    return model, history

# ===============================
# 11. INFERENCE & SUBMISSION
# ===============================
def create_submission(model, test_img_dir, output_file='submission.csv'):
    """Create submission file"""
    print("\n" + "=" * 70)
    print("GENERATING PREDICTIONS")
    print("=" * 70)
    
    test_img_paths = sorted(glob(os.path.join(test_img_dir, '*')))
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    test_img_paths = [p for p in test_img_paths if os.path.splitext(p)[1].lower() in valid_extensions]
    
    print(f"Found {len(test_img_paths)} test images")
    
    submission_data = []
    
    for i, img_path in enumerate(test_img_paths):
        if (i + 1) % 50 == 0:
            print(f"Progress: {(i+1)/len(test_img_paths)*100:.1f}% ({i+1}/{len(test_img_paths)})")
        
        # Load image
        img = load_image(img_path)
        
        # Predict with TTA
        if USE_TTA:
            pred_mask = predict_with_tta(model, img)
        else:
            pred_mask = model.predict(np.expand_dims(img, axis=0), verbose=0)[0, :, :, 0]
        
        # Threshold
        pred_mask_binary = (pred_mask > 0.5).astype(np.uint8)
        
        # Post-process
        pred_mask_binary = post_process_mask(pred_mask_binary)
        
        # Convert to RLE
        rle = mask2rle(pred_mask_binary)
        
        # Get image ID
        img_id = os.path.basename(img_path).replace('.jpg', '').replace('.png', '')
        img_id = img_id + '_segmentation'
        
        submission_data.append([img_id, rle])
    
    # Save submission
    submission_df = pd.DataFrame(submission_data, columns=['ID', 'Predicted_Mask'])
    submission_df.to_csv(output_file, index=False)
    
    print("\n" + "=" * 70)
    print("SUBMISSION COMPLETED!")
    print("=" * 70)
    print(f"Predictions: {len(submission_df)}")
    print(f"File saved: {output_file}")
    
    return submission_df

In [ ]:
# ===============================
# 12. MAIN EXECUTION
# ===============================
if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("IMPROVED SKIN LESION SEGMENTATION PIPELINE")
    print("Key Improvements:")
    print("  - Fixed NaN: Mixed precision ON but loss in float32")
    print("  - Attention UNet architecture")
    print("  - Weighted + Focal Tversky loss")
    print("  - Enhanced augmentation (Albumentations)")
    print("  - Cosine LR schedule with warmup")
    print("  - Dropout + L2 regularization")
    print("  - Improved post-processing")
    print("=" * 70 + "\n")
    
    try:
        # Train
        model, history = main_training()
        
        # Create submission
        submission_df = create_submission(model, TEST_IMG_PATH, 'submission_improved.csv')
        
        print("\n" + "=" * 70)
        print("PIPELINE COMPLETED SUCCESSFULLY!")
        print("=" * 70)
        print("\nNext steps:")
        print("1. Download 'submission_improved.csv'")
        print("2. Submit to Kaggle")
        print("3. Expected improvement over baseline!")
        print("=" * 70 + "\n")
        
    except Exception as e:
        print(f"\nERROR: {str(e)}")
        raise